# EQG MORPH MVP — ibm_fez

**Reviewed runner template, not a new experiment.** This notebook preserves the MORPH → QAOA MaxCut → AR contract workflow. It does not apply DD or E1. The high-friction connected path is the reference heuristic, not a neutral baseline. Original executed notebook snapshot is retained at repository root.

Hardware mode is disabled by default. Review the backend-specific shot/runtime estimate and explicitly set `CONFIRM_SPEND=True` only when authorizing a QPU run. Saved outputs from earlier execution have been cleared so they cannot be mistaken for this reviewed template.


## 0. Install


In [ ]:
!pip install -q --upgrade "numpy>=2.2.0" "qiskit>=1.3.0" "qiskit-ibm-runtime" "qiskit-aer" "scipy"


In [ ]:
import numpy, scipy, qiskit, qiskit_ibm_runtime
print("numpy", numpy.__version__, "| scipy", scipy.__version__)
print("qiskit", qiskit.__version__, "| runtime", qiskit_ibm_runtime.__version__)
# Si AttributeError / ufunc → Restart kernel y vuelve desde !pip


## 1. Config de producto (congelar antes de gastar)


In [ ]:
BACKEND_NAME = "ibm_fez"
CONFIRM_SPEND = False          # True solo después de ver estimación de tiempo
MAX_QPU_S = 180.0

SEED = 20261024
EQG_VERSION = "EQG-MORPH-MVP-reviewed-ibm_fez-1.0.1"

N_LOGICAL = 4                  # path MaxCut; subir a 8 si presupuesto alcanza
QAOA_P = 1
SHOTS = 2048

# Contrato METRIC_CONTRACT_v3.1
DELTA_MIN = 0.02
Z_MIN = 2.0
IC95_LOW_MIN = 0.006

# Fricción: mezcla props + char opcional (si subes results_char.json)
CHAR_RESULTS = "1results_char.json"   # opcional; si no está, solo props

SEC_PER_SHOT = None  # backend-specific estimate; None means estimate must be configured
OVERHEAD_S = 45.0  # planning estimate only, not a provider bill

print(EQG_VERSION)
print(f"backend={BACKEND_NAME} N={N_LOGICAL} p={QAOA_P} shots={SHOTS}")
print(f"gate: ΔAR≥{DELTA_MIN} z≥{Z_MIN} IC95_low>{IC95_LOW_MIN}")
print(f"CONFIRM_SPEND={CONFIRM_SPEND}")


## 2. Token


In [ ]:
import os

TOKEN = os.environ.get("QISKIT_IBM_TOKEN") or os.environ.get("IBM_QUANTUM_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("IBM_QUANTUM_TOKEN")
    except Exception:
        TOKEN = None
assert TOKEN, "Define secreto IBM_QUANTUM_TOKEN. No lo pegues en celdas."
print("token: OK (no se imprime)")


## 3. Utilidades de producto


In [ ]:
import json, math, time, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np

RNG = np.random.default_rng(SEED)
OUT = Path("eqg_product_fez_reviewed")
OUT.mkdir(parents=True, exist_ok=True)
N = N_LOGICAL
MAXCUT_OPT = N - 1
EPS = 1e-12

def z_diff(a, sa, b, sb):
    s = math.sqrt(sa**2 + sb**2)
    return float("nan") if s < EPS else (a - b) / s

def ic95_low(delta, se):
    return delta - 1.96 * se

def verdict(delta, se, z):
    low = ic95_low(delta, se)
    if delta >= DELTA_MIN and z >= Z_MIN and low > IC95_LOW_MIN:
        return "PRODUCTO_OK"
    if delta > 0 and (delta < DELTA_MIN or z < Z_MIN):
        return "SENAL_DEBIL"
    return "NO_GO"

def bitstring_cut(bs, n):
    s = str(bs).replace(" ", "")[-n:].zfill(n)
    return sum(1 for i in range(n - 1) if s[i] != s[i + 1])

def ar_from_counts(counts, n, shots=None):
    total = sum(counts.values()) if shots is None else shots
    if total <= 0:
        return 0.0, 1.0
    cuts, ws = [], []
    for bs, c in counts.items():
        cuts.append(bitstring_cut(bs, n))
        ws.append(c)
    cuts = np.asarray(cuts, float)
    p = np.asarray(ws, float)
    p = p / p.sum()
    mean_cut = float(p @ cuts)
    var = float(p @ (cuts ** 2) - mean_cut ** 2) / max(total, 1)
    return mean_cut / max(MAXCUT_OPT, 1), math.sqrt(max(var, 0.0)) / max(MAXCUT_OPT, 1)

def qaoa_path(n, p, gammas, betas):
    from qiskit import QuantumCircuit
    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    for layer in range(p):
        for i in range(n - 1):
            qc.cx(i, i + 1)
            qc.rz(-2 * gammas[layer], i + 1)
            qc.cx(i, i + 1)
        qc.rx(2 * betas[layer], range(n))
    qc.measure(range(n), range(n))
    return qc

GAMMAS = RNG.uniform(0.25, 0.85, size=QAOA_P)
BETAS = RNG.uniform(0.15, 0.45, size=QAOA_P)
FP = hashlib.sha256(json.dumps({
    "v": EQG_VERSION, "backend": BACKEND_NAME, "N": N, "p": QAOA_P, "shots": SHOTS,
    "seed": SEED, "dmin": DELTA_MIN
}, sort_keys=True).encode()).hexdigest()[:16]
print("fingerprint", FP)
print("gammas", GAMMAS, "betas", BETAS)


## 4. Observe + MORPH

Fricción por qubit = readout + error X (props).  
Si existe `results_char.json` del día, **sobrescribe** esos qubits con fricción medida (T2*/eco) — lo mejor de la char E1b sin obligar el archivo.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=TOKEN, instance=None)
del TOKEN
backend = service.backend(BACKEND_NAME)
n_phys = backend.num_qubits
edges = [list(e) for e in backend.coupling_map.get_edges()]
props = backend.properties()

def friction_props(q):
    try:
        ro = float(props.readout_error(q))
    except Exception:
        ro = 0.03
    try:
        xe = float(props.gate_error("x", [q]))
    except Exception:
        xe = 0.001
    return float(min(1.0, 6.0 * ro + 60.0 * xe))

fr = np.array([friction_props(q) for q in range(n_phys)])
observe_source = "backend_props"
char_job = None

# Char opcional (mejor del día si está)
rp = Path(CHAR_RESULTS)
if rp.exists():
    raw = json.loads(rp.read_text())
    char_job = raw.get("job_id")
    for q_str, rec in (raw.get("results") or {}).items():
        q = int(q_str)
        if q >= n_phys:
            continue
        flags = rec.get("flags") or []
        if "CONTROL_FALLA" in flags:
            fr[q] = 0.95
            continue
        t2s = rec.get("T2star_us")
        t2e_flag = rec.get("T2echo_flag")
        f_khz = abs(float(rec.get("ramsey_f_kHz") or 0))
        if t2s is None:
            base = fr[q]
        elif t2s >= 500:
            base = 0.12
        elif t2s >= 100:
            base = 0.18
        elif t2s >= 30:
            base = 0.35
        else:
            base = float(np.clip(0.55 + 0.35 * (15.0 - t2s) / 12.0, 0.45, 0.92))
        if t2e_flag == "FUERA_DE_RANGO":
            base = min(base, 0.20)
        if f_khz > 15:
            base = min(1.0, base + 0.08)
        fr[q] = float(np.clip(base, 0.05, 0.98))
    observe_source = "char+props"

def morph_path(n_logical, fr, edges, maximize=False, max_paths=400):
    adj = defaultdict(list)
    for a, b in edges:
        a, b = int(a), int(b)
        adj[a].append(b); adj[b].append(a)
    found = []
    def dfs(path):
        if len(found) >= max_paths:
            return
        if len(path) == n_logical:
            found.append(list(path)); return
        key = (lambda q: -fr[q]) if maximize else (lambda q: fr[q])
        for nxt in sorted(adj.get(path[-1], []), key=key):
            if nxt not in path:
                dfs(path + [nxt])
    starts = sorted(range(len(fr)), key=(lambda q: -fr[q]) if maximize else (lambda q: fr[q]))
    for s in starts:
        dfs([s])
        if len(found) >= max_paths:
            break
    if not found:
        return list(range(min(n_logical, len(fr))))
    scored = [(float(np.mean([fr[q] for q in p])), p) for p in found]
    scored.sort(key=(lambda x: -x[0]) if maximize else (lambda x: x[0]))
    return scored[0][1]

# Producto: layout EQG = MORPH (mínima fricción)
LAYOUT_EQG = morph_path(N, fr, edges, maximize=False)
# Referencia ingenua (índices bajos conectados / alta fricción) solo para Δ del contrato
LAYOUT_REF = morph_path(N, fr, edges, maximize=True)
if LAYOUT_REF == LAYOUT_EQG:
    LAYOUT_REF = list(range(N))

print("[qpu]", backend.name, "n_phys", n_phys)
print("observe:", observe_source, "char_job:", char_job)
print("LAYOUT_EQG (MORPH)", LAYOUT_EQG, "mean_fr", round(float(np.mean(fr[LAYOUT_EQG])), 4))
print("LAYOUT_REF (ref)  ", LAYOUT_REF, "mean_fr", round(float(np.mean(fr[LAYOUT_REF])), 4))


## 5. Estimación de gasto (no envía nada)


In [ ]:
# 2 layouts × 1 circuito QAOA cada uno
n_circ = 2
if SEC_PER_SHOT is None:
    raise RuntimeError("Set and justify a backend-specific SEC_PER_SHOT before estimating or submitting QPU work.")
est_s = n_circ * SHOTS * SEC_PER_SHOT + OVERHEAD_S
print(f"circuitos: {n_circ} | shots totales: {n_circ * SHOTS:,}")
print(f"estimación QPU: ~{est_s:.0f} s (+ cola)")
print(f"CONFIRM_SPEND={CONFIRM_SPEND} | tope MAX_QPU_S={MAX_QPU_S}")
if est_s > MAX_QPU_S:
    print("WARN: estimación > tope — sube MAX_QPU_S o baja SHOTS/N")


## 6. Ejecutar producto en QPU


In [ ]:
from qiskit import transpile
from qiskit_ibm_runtime import SamplerV2

if not CONFIRM_SPEND:
    raise SystemExit("CONFIRM_SPEND=False — revisa estimación, pon True y re-ejecuta esta celda.")
if est_s > MAX_QPU_S:
    raise RuntimeError(f"Estimación {est_s:.0f}s > MAX_QPU_S={MAX_QPU_S}")

qc = qaoa_path(N, QAOA_P, GAMMAS, BETAS)
t_eqg = transpile(qc, backend=backend, initial_layout=LAYOUT_EQG, optimization_level=1)
t_ref = transpile(qc, backend=backend, initial_layout=LAYOUT_REF, optimization_level=1)

sampler = SamplerV2(mode=backend)
t0 = time.time()
job = sampler.run([(t_eqg, None, SHOTS), (t_ref, None, SHOTS)])
print("job id:", job.job_id())
result = job.result()
elapsed = time.time() - t0
print(f"wall time: {elapsed:.1f}s")

def counts_i(i):
    pub = result[i].data.c
    if hasattr(pub, "get_counts"):
        return {k: int(v) for k, v in pub.get_counts().items()}
    return {str(k): int(v) for k, v in dict(pub).items()}

AR_EQG, se_E = ar_from_counts(counts_i(0), N, SHOTS)
AR_REF, se_R = ar_from_counts(counts_i(1), N, SHOTS)
delta = AR_EQG - AR_REF
se_d = math.sqrt(se_E**2 + se_R**2)
z = z_diff(AR_EQG, se_E, AR_REF, se_R)
VER = verdict(delta, se_d, z)

print(f"AR_EQG={AR_EQG:.4f}±{se_E:.4f}  AR_REF={AR_REF:.4f}±{se_R:.4f}")
print(f"ΔAR={delta:.4f}  SE={se_d:.4f}  z={z:.2f}  IC95_low={ic95_low(delta, se_d):.4f}")
print("VEREDICTO:", VER)

report = {
    "version": EQG_VERSION,
    "fingerprint": FP,
    "backend": BACKEND_NAME,
    "job_id": job.job_id(),
    "observe_source": observe_source,
    "char_job_id": char_job,
    "n_logical": N,
    "qaoa_p": QAOA_P,
    "shots": SHOTS,
    "layout_EQG": [int(x) for x in LAYOUT_EQG],
    "layout_REF": [int(x) for x in LAYOUT_REF],
    "mean_fr_EQG": float(np.mean(fr[LAYOUT_EQG])),
    "mean_fr_REF": float(np.mean(fr[LAYOUT_REF])),
    "AR_EQG": AR_EQG, "se_EQG": se_E,
    "AR_REF": AR_REF, "se_REF": se_R,
    "delta_AR": delta, "se_delta": se_d, "z": z,
    "ic95_low": ic95_low(delta, se_d),
    "veredicto": VER,
    "contract": {"DELTA_MIN": DELTA_MIN, "Z_MIN": Z_MIN, "IC95_LOW_MIN": IC95_LOW_MIN},
    "wall_s": elapsed,
    "note": "Producto único EQG: MORPH layout vs ref. Sin DD. Sin ley E1.",
}
(OUT / "report.json").write_text(json.dumps(report, indent=2, default=float), encoding="utf-8")
(OUT / "summary.txt").write_text(
    f"VERSION={EQG_VERSION}\nJOB={job.job_id()}\nVEREDICTO={VER}\n"
    f"DELTA_AR={delta}\nZ={z}\nAR_EQG={AR_EQG}\nAR_REF={AR_REF}\n"
    f"LAYOUT_EQG={LAYOUT_EQG}\nLAYOUT_REF={LAYOUT_REF}\n",
    encoding="utf-8",
)
print("Saved", OUT / "report.json")


## 7. Cómo leer el veredicto

| Veredicto | Significado |
|-----------|-------------|
| **PRODUCTO_OK** | MORPH mejora AR ≥ +0.02 con z≥2 e IC95_low>0.006 → evidencia de producto en este día/backend |
| **SENAL_DEBIL** | Mejora positiva pero bajo contrato → más shots o N=8 |
| **NO_GO** | Sin ventaja hoy → re-Observe; no empujar DD a ciegas |

La referencia ingenua solo existe para **calcular Δ** del contrato. El producto que se opera es **LAYOUT_EQG (MORPH)**.
